# TalentHub — HR Recruitment and Employee Attrition Analytics
This notebook connects to PostgreSQL using **psycopg2 DB-API**, runs analytical queries, and plots results.

**Data note:** employee attributes and attrition labels come from the first 500 Kaggle rows. Recruitment candidates/applications/interviews are synthetic demo records generated by `python/generate_project_data.py`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "talenthub"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "CHANGE_ME")
)
print("Connected to PostgreSQL")

In [ ]:
# Load the department attrition view with pandas.
dept = pd.read_sql_query("SELECT * FROM vw_department_attrition ORDER BY department_name", conn)
display(dept)

In [ ]:
# Chart 1: attrition rate by department.
plt.figure(figsize=(8,5))
plt.bar(dept["department_name"], dept["attrition_rate_pct"])
plt.title("Employee Attrition Rate by Department")
plt.xlabel("Department")
plt.ylabel("Attrition rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: attrition counts by overtime.
overtime = pd.read_sql_query(
    """SELECT overtime,
              COUNT(*) AS employees,
              COUNT(*) FILTER (WHERE attrition='Yes') AS leavers
       FROM employees GROUP BY overtime ORDER BY overtime""", conn)
display(overtime)
overtime.plot(x="overtime", y=["employees","leavers"], kind="bar", figsize=(7,4))
plt.title("Employee Count and Attrition by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Number of employees")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: employee attrition by tenure group.
tenure = pd.read_sql_query(
    """SELECT CASE WHEN years_at_company < 2 THEN '0-1 years'
                    WHEN years_at_company < 5 THEN '2-4 years'
                    WHEN years_at_company < 10 THEN '5-9 years'
                    ELSE '10+ years' END AS tenure_band,
              COUNT(*) FILTER (WHERE attrition='Yes') AS leavers
       FROM employees GROUP BY tenure_band""", conn)
display(tenure)
plt.figure(figsize=(7,4))
plt.bar(tenure["tenure_band"], tenure["leavers"])
plt.title("Attrition Count by Tenure Band")
plt.xlabel("Tenure")
plt.ylabel("Employees who left")
plt.tight_layout()
plt.show()

In [ ]:
# Recruitment funnel: simulated demonstration records.
funnel = pd.read_sql_query(
    "SELECT current_stage, COUNT(*) AS applications FROM applications GROUP BY current_stage",
    conn)
display(funnel)
plt.figure(figsize=(8,4))
plt.bar(funnel["current_stage"], funnel["applications"])
plt.title("Recruitment Funnel (Synthetic Demonstration Data)")
plt.xlabel("Application stage")
plt.ylabel("Number of applications")
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print("Database connection closed.")